# Multi-agent Matching Experiment

3エージェントの検出シミュレーションを生成し、RoCo-style、Pairwise RRWM、k-wise RRWM の3手法を比較します。

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path):
    """現在位置から親方向へ、src/simulation.py を持つ repo root を探す。"""
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "src" / "simulation.py").exists():
            return path
    return None


repo_dir = find_repo_root(Path.cwd())

# VS Code から Colab kernel を使っている場合は /content にいるため、public repo を clone する。
if repo_dir is None and Path("/content").exists():
    repo_dir = Path("/content/Multi-agent-Matching")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/furuyuu/Multi-agent-Matching.git", str(repo_dir)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=False)

if repo_dir is None or not (repo_dir / "src" / "simulation.py").exists():
    raise FileNotFoundError("Could not find repository root. Check your notebook kernel location.")

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("repo_dir:", Path.cwd())

## Imports

In [ ]:
import numpy as np
from pathlib import Path

from src.simulation import (
    count_detection_labels,
    generate_detections,
    generate_true_scene,
)
from src.visualization import (
    plot_detected_scene,
    plot_pairwise_matching_result,
    plot_true_scene,
)
from src.graph_matching import (
    build_method_comparison_dataframe,
    KWISE_PARAMS,
    ROCO_PARAMS,
    RRWM_PARAMS,
    evaluate_kwise_matching_3agents,
    kwise_rrwm_matching_3agents,
    kwise_to_pairwise_matches,
    print_kwise_matches_with_true_ids,
    print_matches_with_true_ids,
    print_method_comparison_table,
    run_pairwise_rrwm_all_pairs,
    run_roco_all_pairs,
)
from src.roco_iterative import (
    ROCO_ITERATIVE_PARAMS,
    build_estimated_object_position_dataframe,
    build_agent_pose_error_dataframe,
    build_object_pose_error_dataframe,
    build_pose_error_summary_dataframe,
    evaluate_iterative_roco_pose_errors,
    build_roco_iteration_dataframe,
    run_iterative_roco_pose_adjustment,
)

agent_pairs = [(0, 1), (0, 2), (1, 2)]

## Parameters

In [ ]:
# ここを変更すると、notebook 側で各手法のハイパーパラメータを調整できます。
# src.graph_matching の *_PARAMS はデフォルト値として使い、実験では copy した辞書を渡します。

roco_params = ROCO_PARAMS.copy()
roco_params.update(
    {
        "tau2": 6.0,
        "tau1": 0.3,
        "lambda_dist": 1.0,
        "neighbor_radius": 15.0,
    }
)

rrwm_params = RRWM_PARAMS.copy()
rrwm_params.update(
    {
        "candidate_radius": 6.0,
        "unary_weight": 2.0,
        "pairwise_weight": 1.0,
        "sigma_pos": 2.0,
        "sigma_yaw_deg": 15.0,
        "sigma_edge": 1.5,
        "max_iter": 10000,
        "alpha": 0.2,
        "beta": 30.0,
        "score_threshold": 0.02,
    }
)

kwise_params = KWISE_PARAMS.copy()
kwise_params.update(
    {
        "candidate_radius": 6.0,
        "unary_weight": 2.0,
        "pairwise_weight": 1.0,
        "sigma_pos": 2.0,
        "sigma_yaw_deg": 15.0,
        "sigma_edge": 1.5,
        "max_iter": 10000,
        "alpha": 0.2,
        "beta": 30.0,
        "score_threshold": 0.02,
    }
)

print("roco_params:", roco_params)
print("rrwm_params:", rrwm_params)
print("kwise_params:", kwise_params)

roco_iterative_params = ROCO_ITERATIVE_PARAMS.copy()
roco_iterative_params.update(
    {
        "max_iter": 10,
        "damping": 0.8,
        "pose_tol": 1e-3,
        "min_matches_for_pose": 2,
    }
)

print("roco_iterative_params:", roco_iterative_params)

## Generate Simulation

In [ ]:
def get_output_dir():
    """Colab kernel では Google Drive、ローカル kernel では repo root に保存します。"""
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        output_dir = Path("/content/drive/MyDrive/Multi-agent-Matching/figures")
    except ModuleNotFoundError:
        output_dir = Path.cwd()

    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


# 以前の Colab notebook と同じように、1つの rng をシーン生成と検出生成で共有します。
experiment_seed = 7
output_dir = get_output_dir()
rng = np.random.default_rng(experiment_seed)

print("experiment_seed:", experiment_seed)
print("output_dir:", output_dir)

agents_gt, objects_gt = generate_true_scene(
    num_agents=3,
    num_objects=12,
    rng=rng,
)

agent_pose_est, detections_by_agent = generate_detections(
    agents_gt,
    objects_gt,
    sensing_range=32.0,
    fov_deg=150.0,
    detection_prob=0.85,
    object_pos_noise_std=0.7,
    object_yaw_noise_std_deg=4.0,
    agent_pos_noise_std=1.0,
    agent_yaw_noise_std_deg=3.0,
    num_outliers_per_agent=2,
    rng=rng,
)

for agent_id, (total, inliers, outliers) in count_detection_labels(detections_by_agent).items():
    print(
        f"Agent {agent_id}: detections={total}, "
        f"inliers={inliers}, outliers={outliers}"
    )

In [ ]:
plot_true_scene(
    agents_gt,
    objects_gt,
    save_path=output_dir / f"True_scene_seed_{experiment_seed}.png",
)
plot_detected_scene(
    agents_gt,
    agent_pose_est,
    detections_by_agent,
    save_path=output_dir / f"Observed_scene_seed_{experiment_seed}.png",
)

## Run Matching Methods

In [ ]:
roco_results = run_roco_all_pairs(
    detections_by_agent,
    agent_pairs,
    roco_params,
)

rrwm_results = run_pairwise_rrwm_all_pairs(
    detections_by_agent,
    agent_pairs,
    rrwm_params,
)

dets_0 = detections_by_agent[0]
dets_1 = detections_by_agent[1]
dets_2 = detections_by_agent[2]

kwise_matches, kwise_x, kwise_K, kwise_candidates = kwise_rrwm_matching_3agents(
    dets_0,
    dets_1,
    dets_2,
    **kwise_params,
)

kwise_eval = evaluate_kwise_matching_3agents(
    dets_0,
    dets_1,
    dets_2,
    kwise_matches,
)

print("k-wise evaluation:", kwise_eval)

## Matching Details

In [ ]:
for i, j in agent_pairs:
    print_matches_with_true_ids(
        f"RoCo-style matching: Agent {i} - Agent {j}",
        detections_by_agent[i],
        detections_by_agent[j],
        roco_results[(i, j)]["matches"],
    )
    print("evaluation:", roco_results[(i, j)]["evaluation"])

for i, j in agent_pairs:
    print_matches_with_true_ids(
        f"Pairwise RRWM matching: Agent {i} - Agent {j}",
        detections_by_agent[i],
        detections_by_agent[j],
        rrwm_results[(i, j)]["matches"],
    )
    print("evaluation:", rrwm_results[(i, j)]["evaluation"])

print_kwise_matches_with_true_ids(
    "k-wise MGM style RRWM matching: Agent 0 - 1 - 2",
    dets_0,
    dets_1,
    dets_2,
    kwise_matches,
)
print("evaluation:", kwise_eval)

## Comparison Table

In [ ]:
comparison_df = build_method_comparison_dataframe(
    agent_pairs=agent_pairs,
    roco_results=roco_results,
    rrwm_results=rrwm_results,
    kwise_matches=kwise_matches,
    detections_by_agent=detections_by_agent,
)

comparison_df.style.format(
    {
        "precision": "{:.3f}",
        "recall": "{:.3f}",
    }
).hide(axis="index")

## Matching Visualization

In [ ]:
for i, j in agent_pairs:
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        roco_results[(i, j)]["matches"],
        agent_i=i,
        agent_j=j,
        title=f"RoCo-style matching result: Agent {i} - Agent {j}",
        params=roco_params,
        save_path=output_dir / f"RoCo_{i}{j}_seed_{experiment_seed}.png",
    )

In [ ]:
for i, j in agent_pairs:
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        rrwm_results[(i, j)]["matches"],
        agent_i=i,
        agent_j=j,
        title=f"Pairwise RRWM matching result: Agent {i} - Agent {j}",
        params=rrwm_params,
        save_path=output_dir / f"Pairwise_{i}{j}_seed_{experiment_seed}.png",
    )

In [ ]:
for i, j in agent_pairs:
    pairwise_from_kwise = kwise_to_pairwise_matches(
        kwise_matches,
        pair=(i, j),
    )
    plot_pairwise_matching_result(
        detections_by_agent[i],
        detections_by_agent[j],
        pairwise_from_kwise,
        agent_i=i,
        agent_j=j,
        title=f"k-wise MGM result shown as pairwise: Agent {i} - Agent {j}",
        params=kwise_params,
        save_path=output_dir / f"kwise_{i}{j}_seed_{experiment_seed}.png",
    )

## Iterative RoCo Pose Adjustment

ここからは既存の matching-only 比較とは別に、RoCo-style matching と pose adjustment を交互に実行します。

In [ ]:
iterative_roco_result = run_iterative_roco_pose_adjustment(
    detections_by_agent=detections_by_agent,
    agent_pose_est=agent_pose_est,
    agent_pairs=agent_pairs,
    roco_params=roco_params,
    anchor_agent_id=0,
    **roco_iterative_params,
)

iterative_roco_df = build_roco_iteration_dataframe(iterative_roco_result)
iterative_roco_df.style.format(
    {
        "max_pose_delta": "{:.4f}",
        "precision": "{:.3f}",
        "recall": "{:.3f}",
    }
).hide(axis="index")

In [ ]:
print("Initial agent poses")
for agent_id, pose in agent_pose_est.items():
    print(f"  A{agent_id}: x={pose[0]:.3f}, y={pose[1]:.3f}, theta={np.rad2deg(pose[2]):.3f} deg")

print("\nAdjusted agent poses")
for agent_id, pose in iterative_roco_result["agent_poses"].items():
    print(f"  A{agent_id}: x={pose[0]:.3f}, y={pose[1]:.3f}, theta={np.rad2deg(pose[2]):.3f} deg")

print("\nEstimated object positions:", len(iterative_roco_result["object_positions"]))

In [ ]:
estimated_object_df = build_estimated_object_position_dataframe(iterative_roco_result)
estimated_object_df.style.format(
    {
        "x_est": "{:.3f}",
        "y_est": "{:.3f}",
    }
).hide(axis="index")

In [ ]:
pose_error_result = evaluate_iterative_roco_pose_errors(
    iterative_roco_result,
    agents_gt,
    objects_gt,
)

pose_error_summary_df = build_pose_error_summary_dataframe(pose_error_result)
pose_error_summary_df.style.format(
    {
        "mean_position_error": "{:.3f}",
        "rmse_position_error": "{:.3f}",
        "mean_yaw_error": "{:.4f}",
        "mean_yaw_error_deg": "{:.3f}",
        "mean_pose_error": "{:.3f}",
    }
).hide(axis="index")


In [ ]:
agent_pose_error_df = build_agent_pose_error_dataframe(pose_error_result)
object_pose_error_df = build_object_pose_error_dataframe(pose_error_result)

display(
    agent_pose_error_df.style.format(
        {
            "x_est": "{:.3f}",
            "y_est": "{:.3f}",
            "theta_est": "{:.4f}",
            "x_true": "{:.3f}",
            "y_true": "{:.3f}",
            "theta_true": "{:.4f}",
            "position_error": "{:.3f}",
            "yaw_error": "{:.4f}",
            "yaw_error_deg": "{:.3f}",
            "pose_error": "{:.3f}",
        }
    ).hide(axis="index")
)
display(
    object_pose_error_df.style.format(
        {
            "x_est": "{:.3f}",
            "y_est": "{:.3f}",
            "theta_est": "{:.4f}",
            "x_true": "{:.3f}",
            "y_true": "{:.3f}",
            "theta_true": "{:.4f}",
            "position_error": "{:.3f}",
            "yaw_error": "{:.4f}",
            "yaw_error_deg": "{:.3f}",
            "pose_error": "{:.3f}",
        }
    ).hide(axis="index")
)


In [ ]:
adjusted_detections_by_agent = iterative_roco_result["detections_by_agent"]
adjusted_agent_pose_est = iterative_roco_result["agent_poses"]

plot_detected_scene(
    agents_gt,
    adjusted_agent_pose_est,
    adjusted_detections_by_agent,
    save_path=output_dir / f"Iterative_RoCo_adjusted_scene_seed_{experiment_seed}.png",
)

for i, j in agent_pairs:
    plot_pairwise_matching_result(
        adjusted_detections_by_agent[i],
        adjusted_detections_by_agent[j],
        iterative_roco_result["pairwise_matches"][(i, j)],
        agent_i=i,
        agent_j=j,
        title=f"Iterative RoCo result: Agent {i} - Agent {j}",
        params={**roco_params, **roco_iterative_params},
        save_path=output_dir / f"Iterative_RoCo_{i}{j}_seed_{experiment_seed}.png",
    )